# Covariates

Extracts static and dynamic environmental covariates for the model. Static covariates are time-invariant. Dynamic covariates (NDVI, LST night, rainfall, tree cover) are extracted as annual composites for each year with occurrence records (2000–2024), plus a long-term mean reference composite (2000–2024) for the prediction surface MaxEnt projects onto.

All layers exported at 1km resolution, EPSG:4326, to Google Drive (`sudan_enm_covariates`).

In [9]:
import ee
import geemap

ee.Initialize(project="sudan-enm")

study_extent = ee.Geometry.Rectangle([21.5, 8.5, 39, 22.5])

# Years with occurrence records
occ_years = list(range(2000, 2017)) + [2018, 2020, 2022, 2024]

# Season months for seasonal composites
wet_months = [6, 7, 8, 9, 10]        # June–October
dry_months = [11, 12, 1, 2, 3, 4, 5]  # November–May

# Common export parameters
export_params = {
    'folder': 'sudan_enm_covariates',
    'region': study_extent,
    'scale': 1000,
    'crs': 'EPSG:4326',
    'maxPixels': 1e9
}

## Static Covariates
### Elevation & Slope
From NASA SRTM 30m, resampled to 1km. Slope derived in degrees.

In [10]:
# Load SRTM and derive slope
srtm = ee.Image("USGS/SRTMGL1_003").clip(study_extent)
slope = ee.Terrain.slope(srtm)

# Resample to 1km (native is 30m, so ~1089 input pixels per output)
srtm_1km = srtm.reduceResolution(
    reducer=ee.Reducer.mean(),
    maxPixels = 2048
).reproject(crs="EPSG:4326", scale=1000)

slope_1km = slope.reduceResolution(
    reducer=ee.Reducer.mean(),
    maxPixels=2048
).reproject(crs='EPSG:4326', scale=1000)

# Export elevation
ee.batch.Export.image.toDrive(
    image=srtm_1km,
    description='elevation_1km',
    **export_params
).start()

# Export slope
ee.batch.Export.image.toDrive(
    image=slope_1km,
    description="slope_1km",
    **export_params
).start()

print("Elevation and slope exports started.")

Elevation and slope exports started.


### Distance to Rivers
From HydroSHEDS flow accumulation (15 arc-second, ~500m). Threshold of 500 upstream cells includes seasonal flows (khors), which are ecologically relevant for *P. orientalis* habitat.

In [11]:
# Load flow accumulation and identify river pixels
flow_acc = ee.Image('WWF/HydroSHEDS/15ACC').clip(study_extent)
rivers = flow_acc.gt(500)

# Distance to nearest river pixel
# fastDistanceTransform returns squared distance in pixels
# .sqrt() gives distance in pixels, then multiply by pixel size in meters
river_distance = rivers.fastDistanceTransform(256).sqrt() \
    .multiply(ee.Image.pixelArea().sqrt()) \
    .clip(study_extent)

# Fill gaps where HydroSHEDS has no data (deep Sahara, ~83k cells)
# 100km = ecologically equivalent to "no river anywhere nearby"
river_distance = river_distance.unmask(100000)

# Resample to 1km (native ~500m)
river_distance_1km = river_distance.reduceResolution(
    reducer=ee.Reducer.mean(),
    maxPixels=2048
).reproject(crs='EPSG:4326', scale=1000)

# Export
ee.batch.Export.image.toDrive(
    image=river_distance_1km,
    description='river_distance_1km',
    **export_params
).start()

print("River distance export started.")

River distance export started.


### Vertisols
From HWSD v2.0 (`projects/sat-io/open-datasets/FAO/HWSD_V2_SMU`). Binary layer: 1 = vertisol, 0 = other. WRB2_CODE = 33 corresponds to vertisols, confirmed via the D_WRB2code lookup table in HWSD2.sqlite and point sampling in known vertisol areas (Gedaref, Blue Nile).

In [12]:
# Binary vertisol layer from HWSD v2.0
hwsd2 = ee.Image("projects/sat-io/open-datasets/FAO/HWSD_V2_SMU")
vertisols = hwsd2.select('WRB2_CODE').eq(33).clip(study_extent)

# Reproject to 1km (binary layer, so use mode to preserve 0/1)
vertisols_1km = vertisols.reduceResolution(
    reducer=ee.Reducer.mode(),
    maxPixels=2048
).reproject(crs='EPSG:4326', scale=1000)

# Export
ee.batch.Export.image.toDrive(
    image=vertisols_1km,
    description='vertisols_1km',
    **export_params
).start()

print("Vertisols export started.")

Vertisols export started.


## Dynamic Covariates

Annual composites for each occurrence year (2000–2024), plus a long-term mean (2000–2024) for the prediction surface. Four variables: NDVI, LST night, rainfall, and tree cover.

### NDVI
From MODIS MOD13Q1 (250m, 16-day composites). Scaled by 0.0001. Annual composites are the mean of all 16-day images within each calendar year.

In [13]:
# Load MODIS NDVI and apply scale factor
ndvi_collection = ee.ImageCollection('MODIS/061/MOD13Q1') \
    .select('NDVI') \
    .filterBounds(study_extent)

# Grab the native MODIS projection once, from the first image
modis_proj = ndvi_collection.first().projection()

def scale_ndvi(image):
    return image.multiply(0.0001).copyProperties(image, ['system:time_start'])

ndvi_scaled = ndvi_collection.map(scale_ndvi)

# Seasonal filter — handles dry season wrapping across year boundary
def filter_season(collection, months):
    if 12 in months and 1 in months:
        return collection.filter(
            ee.Filter.Or(
                ee.Filter.calendarRange(11, 12, 'month'),
                ee.Filter.calendarRange(1, 5, 'month')
            )
        )
    else:
        return collection.filter(
            ee.Filter.calendarRange(min(months), max(months), 'month')
        )

# Aggregate a composite to 1km: reattach native projection, reduce, reproject
def to_1km(composite):
    return composite.setDefaultProjection(modis_proj) \
        .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=2048) \
        .reproject(crs='EPSG:4326', scale=1000) \
        .clip(study_extent)

# Annual and seasonal composites for a given year
def ndvi_composites(year):
    year_num = ee.Number(year)
    start = ee.Date.fromYMD(year_num, 1, 1)
    end = ee.Date.fromYMD(year_num.add(1), 1, 1)
    year_imgs = ndvi_scaled.filterDate(start, end)

    annual = to_1km(year_imgs.mean())
    wet = to_1km(filter_season(year_imgs, wet_months).mean())
    dry = to_1km(filter_season(year_imgs, dry_months).mean())

    return {'annual': annual, 'wet': wet, 'dry': dry}

# Export all three composites for each occurrence year
for year in occ_years:
    composites = ndvi_composites(year)
    for season, image in composites.items():
        ee.batch.Export.image.toDrive(
            image=image,
            description=f'ndvi_{season}_{year}_1km',
            **export_params
        ).start()

# Long-term means for prediction surface
ndvi_all = ndvi_scaled.filterDate('2000-01-01', '2025-01-01')

for season_name, months in [('annual', None), ('wet', wet_months), ('dry', dry_months)]:
    imgs = ndvi_all if months is None else filter_season(ndvi_all, months)
    mean_img = to_1km(imgs.mean())
    ee.batch.Export.image.toDrive(
        image=mean_img,
        description=f'ndvi_{season_name}_mean_2000_2024_1km',
        **export_params
    ).start()

print(f"NDVI exports started: {len(occ_years) * 3} annual/wet/dry + 3 long-term means")

NDVI exports started: 63 annual/wet/dry + 3 long-term means


### LST
From MODIS MOD11A2 (1km, 8-day composites). Nighttime LST is ecologically relevant since *P. orientalis* is a nocturnal vector. Scale factor 0.02, converted from Kelvin to Celsius.

In [14]:
# Load MODIS LST day and night
lst_bands = {
    'day': 'LST_Day_1km',
    'night': 'LST_Night_1km'
}

def scale_lst(image):
    return image.multiply(0.02).subtract(273.15) \
        .copyProperties(image, ['system:time_start'])

# Annual and seasonal composites for a given year and band
def lst_composites(year, band):
    year_num = ee.Number(year)
    start = ee.Date.fromYMD(year_num, 1, 1)
    end = ee.Date.fromYMD(year_num.add(1), 1, 1)
    year_imgs = ee.ImageCollection('MODIS/061/MOD11A2') \
        .select(band) \
        .filterBounds(study_extent) \
        .filterDate(start, end) \
        .map(scale_lst)

    annual = year_imgs.mean().clip(study_extent)
    wet = filter_season(year_imgs, wet_months).mean().clip(study_extent)
    dry = filter_season(year_imgs, dry_months).mean().clip(study_extent)

    return {'annual': annual, 'wet': wet, 'dry': dry}

# Export annual/wet/dry for day and night, for each occurrence year
for year in occ_years:
    for label, band in lst_bands.items():
        composites = lst_composites(year, band)
        for season, image in composites.items():
            ee.batch.Export.image.toDrive(
                image=image,
                description=f'lst_{label}_{season}_{year}_1km',
                **export_params
            ).start()

# Long-term means for prediction surface
for label, band in lst_bands.items():
    all_imgs = ee.ImageCollection('MODIS/061/MOD11A2') \
        .select(band) \
        .filterBounds(study_extent) \
        .filterDate('2000-01-01', '2025-01-01') \
        .map(scale_lst)

    for season_name, months in [('annual', None), ('wet', wet_months), ('dry', dry_months)]:
        imgs = all_imgs if months is None else filter_season(all_imgs, months)
        mean_img = imgs.mean().clip(study_extent)
        ee.batch.Export.image.toDrive(
            image=mean_img,
            description=f'lst_{label}_{season_name}_mean_2000_2024_1km',
            **export_params
        ).start()

total = len(occ_years) * 6 + 6
print(f"LST exports started: {total} total (day+night × annual/wet/dry × {len(occ_years)} years + 6 means)")

LST exports started: 132 total (day+night × annual/wet/dry × 21 years + 6 means)


### Rainfall
From CHIRPS Daily v2.0. Annual composites are the sum of daily rainfall within each calendar year (mm/year). No seasonal split — annual totals are the ecologically meaningful metric for rainfall.

In [15]:
# Load CHIRPS daily rainfall
rainfall_collection = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
    .filterBounds(study_extent)

# Annual total for a given year
def rainfall_annual(year):
    year_num = ee.Number(year)
    start = ee.Date.fromYMD(year_num, 1, 1)
    end = ee.Date.fromYMD(year_num.add(1), 1, 1)
    return rainfall_collection.filterDate(start, end).sum().clip(study_extent)

# Export annual totals for each occurrence year
for year in occ_years:
    image = rainfall_annual(year)
    ee.batch.Export.image.toDrive(
        image=image,
        description=f'rainfall_{year}_1km',
        **export_params
    ).start()

# Long-term mean annual rainfall for prediction surface
all_years = list(range(2000,2025)) # 2000 through 2024 inclusive
annual_sums = ee.ImageCollection([rainfall_annual(y) for y in all_years])
rainfall_mean = annual_sums.mean().clip(study_extent)

ee.batch.Export.image.toDrive(
    image=rainfall_mean,
    description='rainfall_mean_2000_2024_1km',
    **export_params
).start()

print(f"Rainfall exports started: {len(occ_years)} annual + 1 long-term mean")

Rainfall exports started: 21 annual + 1 long-term mean


### Tree Cover
From Hansen Global Forest Change (v1.13, 2025 release). Dynamic World only starts in 2015, which misses 15 of 21 occurrence years. Hansen provides a year-2000 baseline tree cover percentage and annual forest loss layers through 2024, allowing reconstruction of tree cover for any year in the study period.

Tree cover for year *t* = year-2000 baseline, with pixels that experienced forest loss through year *t* set to 0. Values are percentage canopy cover (0–100). Loss is binary per pixel in Hansen, so a lost pixel drops to 0 rather than a partial reduction — appropriate here since presence of tree cover matters more than gradation for Acacia-Balanites woodland.

In [16]:
# Load Hansen Global Forest Change (v1.13, 2025 release)
hansen = ee.Image('UMD/hansen/global_forest_change_2025_v1_13').clip(study_extent)

# Year-2000 baseline tree cover (%)
treecover_2000 = hansen.select('treecover2000')

# Loss year band: pixel value = year of loss (1–24, meaning 2001–2024), 0 = no loss
lossyear = hansen.select('lossyear')

# Reconstruct tree cover for a given year, aggregated to 1km
def treecover_for_year(year):
    if year == 2000:
        reconstructed = treecover_2000
    else:
        # Cap at 2024 (latest available loss year in the 2025 release)
        loss_year_cap = min(year, 2024)
        # Loss year band uses 1 = 2001, 2 = 2002, etc.
        year_code = loss_year_cap - 2000
        # Cumulative loss: any pixel lost in years 1 through year_code
        cumulative_loss = lossyear.gt(0).And(lossyear.lte(year_code))
        # Set lost pixels to 0, keep baseline elsewhere
        reconstructed = treecover_2000.where(cumulative_loss, 0)
    # Aggregate 30m native to 1km via mean (matches elevation/NDVI handling)
    return reconstructed.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=2048
    ).reproject(crs='EPSG:4326', scale=1000).clip(study_extent)

# Export annual tree cover for each occurrence year (model fitting uses occurrence years)
for year in occ_years:
    image = treecover_for_year(year)
    ee.batch.Export.image.toDrive(
        image=image,
        description=f'treecover_{year}_1km',
        **export_params
    ).start()

# Long-term mean for prediction surface (all years 2000–2024, not just occurrence years)
all_years = list(range(2000, 2025))  # 2000–2024 inclusive
tc_images = ee.ImageCollection([treecover_for_year(y) for y in all_years])
treecover_mean = tc_images.mean().clip(study_extent)

ee.batch.Export.image.toDrive(
    image=treecover_mean,
    description='treecover_mean_2000_2024_1km',
    **export_params
).start()

print(f"Tree cover exports started: {len(occ_years)} annual + 1 long-term mean (mean over {len(all_years)} years)")

Tree cover exports started: 21 annual + 1 long-term mean (mean over 25 years)


## Export Summary

### Static (4 rasters)
- `elevation_1km`
- `slope_1km`
- `river_distance_1km`
- `vertisols_1km`

### Dynamic — annual composites per occurrence year (21 years × variable)
- `ndvi_{annual/wet/dry}_{year}_1km` — 63 rasters
- `lst_day_{annual/wet/dry}_{year}_1km` — 63 rasters
- `lst_night_{annual/wet/dry}_{year}_1km` — 63 rasters
- `rainfall_{year}_1km` — 21 rasters
- `treecover_{year}_1km` — 21 rasters

### Long-term means for prediction surface (11 rasters)
- `ndvi_{annual/wet/dry}_mean_2000_2024_1km`
- `lst_day_{annual/wet/dry}_mean_2000_2024_1km`
- `lst_night_{annual/wet/dry}_mean_2000_2024_1km`
- `rainfall_mean_2000_2024_1km`
- `treecover_mean_2000_2024_1km`

**Total: 246 rasters** to `Google Drive > sudan_enm_covariates`